In [1]:
#@title student data
from IPython.display import Javascript
colab_base = "https://colab.research.google.com/drive/1BVBGdNkEuqIgFp6XxmaV9NJy5c2sX3ln?usp=sharing"
Grupa = "niedotyczy" # @param ["niedotyczy", "13_45","15_30","17_15"]
Student_ID = "473616" # @param {"type":"string"}
Link_to_this_colab = "https://github.com/pawelFelcyn/nlp" # @param {"type":"string"}
Mail = "" # @param {"type":"string","placeholder":"Optional"}

if Link_to_this_colab == colab_base:
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))

#Task D01

## 🧩 BERT-style Masked Language Modeling — przygotowanie masek

W tym zadaniu zasymulujesz **krok przygotowania danych do BERT-owego MLM**:
dla zadanej sekwencji tokenów wybierzesz te, które będą maskowane, i zbudujesz odpowiednie wektory.

---

## 🎯 Cel

- zrozumieć, jak BERT wybiera tokeny do zadania Masked Language Modeling,
- poćwiczyć operacje na tablicach `numpy` (indeksy, maski, losowanie),
- przygotować docelowe tensory: `input_ids_masked` i `labels`.

---

## 📘 Kontekst

W klasycznym BERT-cie:

- wybieramy ok. **15% tokenów** do przewidywania,
- z tych tokenów:
  - 80% zamieniamy na token specjalny `[MASK]`,
  - 10% na losowy token ze słownika,
  - 10% zostawiamy bez zmian,
- **loss liczymy tylko na pozycjach wybranych do maskowania**.

W tym zadaniu pracujemy na uproszczonym przykładzie z małym słownikiem.

---

## ✅ Twoje zadanie

1. W komórce `dane` masz zadane:
   - `vocab_size`,
   - `MASK_ID`,
   - `input_ids` — wektor `np.array` z ID tokenów,
   - `mask_prob` — prawdopodobieństwo, że dany token trafi do puli maskowanej.
2. Zaimplementuj funkcję:

```python
def bert_mlm_mask(input_ids, mask_prob, vocab_size, MASK_ID, seed):
    ...
```

która:

- używa `np.random.seed(seed)` dla powtarzalności,
- losuje wektor `mask_flags` tej samej długości co `input_ids` z rozkładu Bernoulliego o parametrze `mask_prob`,
- na pozycjach, gdzie `mask_flags == 1`:
  - z prawdopodobieństwem 0.8 wstawia `MASK_ID`,
  - z prawdopodobieństwem 0.1 losuje **inny** token ID z zakresu `[0, vocab_size)`,
  - z prawdopodobieństwem 0.1 zostawia token bez zmian,
- tworzy wektor `labels`:
  - `labels[i] = input_ids[i]` dla pozycji maskowanych,
  - `labels[i] = -100` dla pozycji niemaskowanych (ignorowane w loss).
3. Funkcja powinna zwrócić krotkę `(input_ids_masked, labels)`.

4. W komórce `Answer`:

- wywołaj tę funkcję z podanymi danymi,
- zbuduj `final_answer` jako string:

```text
input_ids_masked:ID0,ID1,...;labels:L0,L1,...
```

bez spacji.

Przykład formatu:

```text
input_ids_masked:101,103,200;labels:-100,30522,-100
```

`final_answer` **musi być dokładnie takim stringiem**, żeby serwer mógł go poprawnie rozparsować.


In [2]:
#@title dane

import numpy as np

vocab_size = 30
MASK_ID = 29

# prosta sekwencja "tokenów"
input_ids = np.array([5, 7, 11, 13, 17, 19], dtype=int)

mask_prob = 0.4
seed = 123


In [3]:
#@title code

import numpy as np

def bert_mlm_mask(input_ids, mask_prob, vocab_size, MASK_ID, seed):
    """Symulacja BERT-owego MLM: zwraca (input_ids_masked, labels).

    input_ids: 1D np.array z ID tokenów
    mask_prob: prawdopodobieństwo wejścia tokenu do puli maskowanej
    vocab_size: rozmiar słownika
    MASK_ID: ID tokenu [MASK]
    seed: ziarno RNG
    """
    np.random.seed(seed)

    masked = input_ids.copy()

    mask_flags = np.random.rand(len(input_ids)) < mask_prob

    labels = np.full_like(input_ids, -100)
    labels[mask_flags] = input_ids[mask_flags]

    rnd = np.random.rand(len(input_ids))

    mask80 = mask_flags & (rnd < 0.8)
    masked[mask80] = MASK_ID

    rand10 = mask_flags & (rnd >= 0.8) & (rnd < 0.9)
    if rand10.any():
        random_tokens = np.random.randint(0, vocab_size, size=rand10.sum())
        for i, pos in enumerate(np.where(rand10)[0]):
            while random_tokens[i] == input_ids[pos]:
                random_tokens[i] = np.random.randint(0, vocab_size)
        masked[rand10] = random_tokens


    return masked, labels
    


In [4]:
# @title Answer

input_ids_masked, labels = bert_mlm_mask(input_ids, mask_prob, vocab_size, MASK_ID, seed)

ids_str = ",".join(str(int(x)) for x in input_ids_masked)
labels_str = ",".join(str(int(x)) for x in labels)

final_answer = f"input_ids_masked:{ids_str};labels:{labels_str}"
print(final_answer)


input_ids_masked:5,29,29,13,17,19;labels:-100,7,11,-100,-100,-100


In [5]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD01"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


input_ids_masked:5,29,29,13,17,19;labels:-100,7,11,-100,-100,-100
65 znaków
Wysłano!
Punkty za zadanie: 1


#Task D02

## 🧩 BERT MLM — liczenie straty krzyżowo-entropijnej dla kilku pozycji

W tym zadaniu policzysz **średnią stratę cross-entropy** dla maskowanych pozycji w zadaniu MLM.

---

## 🎯 Cel

- zrozumieć, jak liczona jest strata w BERT-owym Masked Language Modeling,
- poćwiczyć log-softmax i wybieranie właściwych logitów.

---

## 📘 Kontekst

Dla danej pozycji maskowanej mamy:

- wektor logitów `logits` o długości `V` (rozmiar słownika),
- indeks poprawnego tokenu `target_id`.

Strata krzyżowo-entropijna (CE) dla jednej pozycji to:

$$
\text{CE} = -\log p_{\text{target}} = -\log\left(\text{softmax}(\text{logits})_{\text{target}}\right).
$$

Dla wielu pozycji liczymy **średnią** z poszczególnych CE.

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - macierz `logits` o kształcie `(num_masked, vocab_size)`,
   - wektor `target_ids` o długości `num_masked`.
2. Zaimplementuj funkcję:

```python
def mlm_cross_entropy(logits, target_ids):
    ...
```

która:

- oblicza softmax po ostatniej osi,
- wybiera prawdopodobieństwa odpowiadające poprawnym indeksom,
- liczy stratę `-log(p_target)` dla każdej pozycji,
- zwraca **średnią** wartość straty jako liczbę zmiennoprzecinkową.

3. W `Answer`:

- policz `loss = mlm_cross_entropy(logits, target_ids)`,
- zaokrąglij do **4 miejsc po przecinku**,
- zapisz w `final_answer` jako string, np.:

```text
0.6931
```


In [6]:
#@title dane

import numpy as np

# num_masked = 3, vocab_size = 5
logits = np.array([
    [2.0, 0.5, -1.0, 0.0, 1.0],
    [0.0, 1.0, 2.0, -1.0, -2.0],
    [-0.5, 0.0, 0.5, 1.0, 1.5],
], dtype=float)

target_ids = np.array([0, 2, 4], dtype=int)


In [7]:
#@title code

import numpy as np

def mlm_cross_entropy(logits, target_ids):
    """Średnia strata CE dla maskowanych pozycji w zadaniu MLM.

    logits: macierz (num_masked, vocab_size)
    target_ids: wektor (num_masked,)
    """
    max_logits = np.max(logits, axis=1, keepdims=True)
    logits_stable = logits - max_logits

    logsumexp = np.log(np.sum(np.exp(logits_stable), axis=1))

    correct_log_probs = logits_stable[np.arange(len(target_ids)), target_ids] - logsumexp

    loss = -np.mean(correct_log_probs)
    return loss


In [8]:
# @title Answer

loss = mlm_cross_entropy(logits, target_ids)
final_answer = f"{loss:.4f}"
print(final_answer)

0.6245


In [9]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD02"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


0.6245
6 znaków
Wysłano!
Punkty za zadanie: 1


#Task D03

## 🧩 Static vs dynamic masking — ile różnych masek?

RoBERTa wprowadza m.in. **dynamic masking**:
maskowane pozycje zmieniają się między przejściami po danych, zamiast być ustalone na stałe jak w BERT-cie.

W tym zadaniu policzysz, ile różnych masek można wytworzyć dla krótkiej sekwencji.

---

## 🎯 Cel

- zrozumieć różnicę między statycznym a dynamicznym maskowaniem,
- policzyć kombinatorycznie liczbę możliwych masek.

---

## 📘 Kontekst

Załóżmy, że:

- mamy sekwencję o długości `n`,
- wybieramy dokładnie `k` pozycji do maskowania,
- maskę reprezentujemy jako wektor długości `n` z wartościami `0` lub `1`,
- `1` oznacza pozycję maskowaną.

Liczba możliwych masek (przy wyborze dokładnie `k` pozycji) to:

$$
\binom{n}{k}.
$$

---

## ✅ Twoje zadanie

1. W `dane` masz podane `n` i `k`.
2. Zaimplementuj funkcję:

```python
def count_masks(n, k):
    ...
```

która zwraca wartość \(\binom{n}{k}\) jako liczbę całkowitą.
Możesz użyć prostej funkcji do liczenia dwumianu Newtona na podstawie iloczynów.

3. W `Answer`:

- policz `masks = count_masks(n, k)`,
- zapisz w `final_answer` string:

```text
masks=LICZBA
```

np.:

```text
masks=10
```


In [10]:
#@title dane

n = 8
k = 3


In [11]:
#@title code

def count_masks(n, k):
    """Zwraca liczbę masek długości n z dokładnie k jedynkami (dwumian Newtona)."""
    if k < 0 or k > n:
        return 0
    k = min(k, n - k)

    result = 1
    for i in range(1, k + 1):
        result = result * (n - (k - i)) // i
    return result


In [12]:
# @title Answer

masks = count_masks(n, k)
final_answer = f"masks={masks}"
print(final_answer)

masks=56


In [13]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD03"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


masks=56
8 znaków
Wysłano!
Punkty za zadanie: 1


#Task D04

## 🧩 Długość sekwencji — porównanie BERT vs RoBERTa

Załóżmy, że:

- model A (np. BERT-base) ma maksymalną długość sekwencji `max_len_A`,
- model B (np. RoBERTa-large) ma maksymalną długość sekwencji `max_len_B`.

Masz dokument o długości `doc_len` tokenów.
Chcesz policzyć, ile segmentów trzeba przygotować, żeby cały dokument został przetworzony przez każdy model.

---

## 🎯 Cel

- policzyć liczbę segmentów (chunków) potrzebnych dla różnych maksymalnych długości sekwencji,
- zobaczyć, jak wydłużenie maksymalnej sekwencji zmniejsza liczbę wywołań modelu.

---

## 📘 Kontekst

Dla danego `max_len` liczba segmentów to:

$$
\text{segments} = \left\lceil \frac{\text{doc_len}}{\text{max_len}} \right\rceil.
$$

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `doc_len`,
   - `max_len_A`,
   - `max_len_B`.
2. Zaimplementuj funkcję:

```python
def num_segments(doc_len, max_len):
    ...
```

która zwraca liczbę segmentów jako int (z zaokrągleniem w górę).
3. W `Answer`:

- policz `seg_A = num_segments(doc_len, max_len_A)`,
- policz `seg_B = num_segments(doc_len, max_len_B)`,
- zapisz w `final_answer` string:

```text
A=LICZBA_A;B=LICZBA_B
```

np.:

```text
A=5;B=3
```


In [14]:
#@title dane

doc_len = 4096
max_len_A = 512   # np. klasyczny BERT
max_len_B = 1024  # np. wariant wydłużony


In [ ]:
#@title code

import math

def num_segments(doc_len, max_len):
    """Zwraca liczbę segmentów potrzebnych, aby pokryć doc_len przy długości max_len."""
    return math.ceil(doc_len / max_len)


In [16]:
# @title Answer

seg_A = num_segments(doc_len, max_len_A)
seg_B = num_segments(doc_len, max_len_B)

final_answer = f"A={seg_A};B={seg_B}"
print(final_answer)

A=8;B=4


In [17]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD04"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


A=8;B=4
7 znaków
Wysłano!
Punkty za zadanie: 1


#Task D05

## 🧩 Klasyfikacja tokenów (NER) na wyjściu enkodera

Załóżmy, że encoder BERT-a zwraca dla każdej pozycji sekwencji wektor,
na podstawie którego prosta warstwa liniowa przewiduje klasy NER.

W tym zadaniu masz już dane **logity** po ostatniej warstwie liniowej
i masz je tylko zamienić na końcowe klasy.

---

## 🎯 Cel

- przećwiczyć wybieranie klasy o największym logicie,
- zamienić indeksy klas na symboliczne etykiety.

---

## 📘 Kontekst

Dla zadania NER mamy np. 3 klasy:

- `0` — `O` (poza nazwą własną),
- `1` — `PER`,
- `2` — `LOC`.

Dla każdej pozycji sekwencji mamy wektor logitów długości 3.
Wybieramy klasę jako argmax po ostatniej osi.

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - macierz `logits` o kształcie `(seq_len, num_classes)`,
   - listę `id2label`, np. `["O", "PER", "LOC"]`.
2. Zaimplementuj funkcję:

```python
def ner_decode(logits, id2label):
    ...
```

która:

- dla każdej pozycji wybiera indeks klasy `argmax`,
- zamienia indeksy na stringi etykiet,
- zwraca listę etykiet `["O", "PER", ...]` długości `seq_len`.
3. W `Answer`:

- policz `labels = ner_decode(logits, id2label)`,
- zapisz w `final_answer` string:

```text
O,PER,O,LOC,...
```

czyli etykiety oddzielone przecinkami, bez spacji.


In [18]:
#@title dane

import numpy as np

logits = np.array([
    [ 2.0,  0.1, -1.0],  # token 0
    [-0.5,  1.2,  0.0],  # token 1
    [ 1.0,  0.0,  0.5],  # token 2
    [-1.0, -0.2,  2.5],  # token 3
], dtype=float)

id2label = ["O", "PER", "LOC"]


In [19]:
#@title code

def ner_decode(logits, id2label):
    """Zwraca listę stringów etykiet NER dla kolejnych tokenów."""
    pred_ids = np.argmax(logits, axis=1)
    return [id2label[i] for i in pred_ids]


In [20]:
# @title Answer

labels = ner_decode(logits, id2label)
final_answer = ",".join(labels)
print(final_answer)

O,PER,O,LOC


In [21]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD05"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


O,PER,O,LOC
11 znaków
Wysłano!
Punkty za zadanie: 1


#Task D06

## 🧩 Kontrastive loss (InfoNCE) dla prostego przykładu

Część modeli enkoderowych (np. w stylu *contrastive learning*) używa straty zbliżonej do **InfoNCE**:
dla danego wektora zapytania `q` mamy jeden pozytywny wektor `k_pos` oraz kilka negatywnych `k_neg`.

W tym zadaniu policzymy prostą wersję takiej straty.

---

## 🎯 Cel

- zrozumieć, jak wygląda prosty kontrastive loss,
- poćwiczyć operacje na iloczynach skalarnych i softmaxie.

---

## 📘 Kontekst

Załóż, że:

- masz zapytanie `q` (wektor 1D),
- masz listę kluczy `K = [k_pos, k_neg1, k_neg2, ...]`,
- liczymy skalarne podobieństwo `s_i = q · K_i`,
- liczymy softmax po wszystkich `s_i`,
- strata to:

$$
\text{loss} = -\log p_{\text{pos}},
$$

czyli minus log-prawdopodobieństwo przypisane **pozytywnemu** kluczowi.

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - wektor `q`,
   - macierz `K` o kształcie `(num_keys, dim)`, gdzie wiersz 0 to `k_pos`,
   - zakładamy, że indeks 0 zawsze odpowiada pozytywnemu kluczowi.
2. Zaimplementuj funkcję:

```python
def info_nce_loss(q, K):
    ...
```

która:

- liczy iloczyny skalarne `s = K @ q`,
- liczy softmax po `s`,
- zwraca `-log(p_pos)` jako liczbę zmiennoprzecinkową.
3. W `Answer`:

- policz `loss = info_nce_loss(q, K)`,
- zaokrąglij do **4 miejsc po przecinku**,
- zapisz w `final_answer` jako string, np.:

```text
0.4213
```


In [22]:
#@title dane

import numpy as np

q = np.array([0.5, 1.0, -0.5], dtype=float)

K = np.array([
    [0.6,  0.9, -0.4],   # pozytywny (index 0)
    [1.0, -0.5,  0.0],   # negatywny
    [0.0,  0.5,  0.5],   # negatywny
], dtype=float)


In [23]:
#@title code

import numpy as np

def info_nce_loss(q, K):
    """Prosty InfoNCE: zakładamy, że pozytywny klucz jest na indeksie 0."""
    logits = K @ q 
    max_logit = np.max(logits)
    logits_stable = logits - max_logit
    exp_logits = np.exp(logits_stable)
    probs = exp_logits / np.sum(exp_logits)
    loss = -np.log(probs[0])

    return loss


In [24]:
# @title Answer

loss = info_nce_loss(q, K)
final_answer = f"{loss:.4f}"
print(final_answer)

0.4468


In [25]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD06"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


0.4468
6 znaków
Wysłano!
Punkty za zadanie: 1


#Task D07

## 🧩 Autoregresyjna generacja (greedy) z prostego modelu dekoderowego

W tym zadaniu zasymulujesz **autoregresyjną generację** w stylu GPT-2,
ale na bardzo prostym, ręcznie zdefiniowanym modelu.

---

## 🎯 Cel

- zrozumieć, jak dekoder generuje kolejne tokeny na podstawie poprzednich,
- przećwiczyć iteracyjne wywoływanie funkcji „modelu” i budowanie sekwencji.

---

## 📘 Kontekst

Załóżmy, że mamy uproszczony model językowy, który:

- dostaje **ostatni wygenerowany token**,
- zwraca **rozkład prawdopodobieństwa** na następny token `p(next_token | last_token)`.

W praktyce GPT-2 używa całej sekwencji jako kontekstu,
ale tutaj upraszczamy do zależności tylko od ostatniego tokenu.

Generacja „greedy”:

1. Zaczynamy od tokenu początkowego `start_id`.
2. W każdej iteracji:
   - bierzemy `last_id` — ID ostatniego tokenu w sekwencji,
   - pobieramy wektor `probs[last_id]` z macierzy prawdopodobieństw,
   - wybieramy token o największym prawdopodobieństwie (`argmax`),
   - dopisujemy go do sekwencji.
3. Kończymy po `max_steps` krokach lub jeśli wygenerujemy `END_ID`.

---

## ✅ Twoje zadanie

1. W komórce `dane` masz:
   - `probs` — macierz `shape = (vocab_size, vocab_size)`, gdzie `probs[i]` to rozkład `p(next | i)`,
   - `start_id` — ID tokenu początkowego,
   - `END_ID` — ID tokenu końca sekwencji,
   - `max_steps` — maksymalna liczba kroków generacji.
2. Zaimplementuj funkcję:

```python
def greedy_generate(probs, start_id, END_ID, max_steps):
    ...
```

która zwraca listę lub wektor `np.array` z kolejnymi tokenami **wraz z tokenem startowym**.
3. W komórce `Answer`:

- wywołaj `seq = greedy_generate(...)`,
- zbuduj `final_answer` jako ciąg ID-ów oddzielonych przecinkami, np.:

```text
1,3,4,4,2
```

bez spacji.


In [26]:
#@title dane

import numpy as np

# Słownik: 0,1,2,3,4 (0 będzie END_ID)
vocab_size = 5
END_ID = 0
start_id = 1
max_steps = 6

# probs[i] = p(next_token | current_token = i)
probs = np.array([
    # next: 0    1    2    3    4
    [0.7, 0.1, 0.1, 0.1, 0.0],  # z 0 (END) teoretycznie nie generujemy dalej
    [0.0, 0.1, 0.6, 0.3, 0.0],  # z 1
    [0.2, 0.0, 0.1, 0.2, 0.5],  # z 2
    [0.5, 0.0, 0.0, 0.2, 0.3],  # z 3
    [0.9, 0.0, 0.1, 0.0, 0.0],  # z 4
], dtype=float)


In [27]:
#@title code

import numpy as np

def greedy_generate(probs, start_id, END_ID, max_steps):
    """Autoregresyjna generacja greedy.

    Zwraca listę ID-ów, zaczynając od start_id, dopisując kolejne tokeny
    aż do osiągnięcia END_ID lub max_steps kroków (bez liczenia tokenu startowego).
    """
    seq = [start_id]
    last_id = start_id

    for _ in range(max_steps):
        # rozkład p(next | last_id)
        next_id = np.argmax(probs[last_id])

        seq.append(next_id)

        # jeśli trafiliśmy END_ID — kończymy generację
        if next_id == END_ID:
            break

        last_id = next_id

    return seq


In [28]:
# @title Answer

seq = greedy_generate(probs, start_id, END_ID, max_steps)
final_answer = ",".join(str(int(x)) for x in seq)
print(final_answer)

1,2,4,0


In [29]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD07"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


1,2,4,0
7 znaków
Wysłano!
Punkty za zadanie: 1


#Task D08

## 🧩 Top-k filtering — przycinanie rozkładu

W tym zadaniu zaimplementujesz **top-k filtering** używany przy samplingu z modeli dekoderowych.

---

## 🎯 Cel

- poćwiczyć sortowanie i przycinanie rozkładu,
- zrozumieć, jak ograniczyć się do `k` najbardziej prawdopodobnych tokenów.

---

## 📘 Kontekst

Dla rozkładu prawdopodobieństwa `p` (np. wektor długości `V`) top-k działa tak:

1. Znajdź `k` największych wartości,
2. Ustaw wszystkie inne wartości na 0,
3. Zrenormalizuj, aby suma znów wynosiła 1.

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - wektor `probs` (prawdopodobieństwa, które sumują się do 1),
   - `k` — ile największych wartości chcemy zachować.
2. Zaimplementuj funkcję:

```python
def top_k_filter(probs, k):
    ...
```

która:

- zwraca nowy wektor `filtered` o tym samym kształcie,
- `filtered[i] = 0` jeśli `probs[i]` nie jest w top-k,
- wynik jest znormalizowany tak, aby jego suma wynosiła 1 (jeśli k > 0).
3. W `Answer`:

- policz `filtered = top_k_filter(probs, k)`,
- zaokrąglij wszystkie wartości do 3 miejsc po przecinku,
- zapisz `final_answer` jako string:

```text
p0,p1,p2,p3,p4
```

bez spacji.


In [30]:
#@title dane

import numpy as np

probs = np.array([0.05, 0.1, 0.6, 0.2, 0.05], dtype=float)
k = 2


In [31]:
#@title code

import numpy as np

def top_k_filter(probs, k):
    """Zwraca rozkład przycięty do k największych wartości, z renormalizacją."""
    probs = probs.copy()

    if k >= len(probs):
        return probs / probs.sum()

    threshold = np.partition(probs, -k)[-k]

    filtered = np.where(probs >= threshold, probs, 0.0)

    s = filtered.sum()
    if s > 0:
        filtered /= s

    return filtered


In [32]:
# @title Answer

filtered = top_k_filter(probs, k)
rounded = [f"{p:.3f}" for p in filtered]
final_answer = ",".join(rounded)
print(final_answer)

0.000,0.000,0.750,0.250,0.000


In [33]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD08"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


0.000,0.000,0.750,0.250,0.000
29 znaków
Wysłano!
Punkty za zadanie: 1


#Task D09

## 🧩 Nucleus (top-p) — budowanie maski

W tym zadaniu zaimplementujesz **krok budowania maski** dla nucleus sampling (top-p).

---

## 🎯 Cel

- nauczyć się wyznaczać minimalny prefiks rozkładu,
- zrozumieć, jak działa próg kumulatywny `p`.

---

## 📘 Kontekst

Dla danego wektora prawdopodobieństw `probs` (załóż, że już **posortowany malejąco**):

1. Liczymy kumulatywną sumę: `cumsum[i] = sum_{j<=i} probs[j]`,
2. Znajdujemy najmniejszy indeks `m`, dla którego `cumsum[m] >= p`,
3. Zatrzymujemy tokeny `0..m`, resztę ignorujemy.

W praktyce nucleus sampling:

- losuje token **tylko** z tego prefiksu `0..m`,
- a następnie renormalizuje prawdopodobieństwa.

W tym zadaniu zrobimy tylko **maskę** prefiksu.

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - wektor `probs_sorted` (posortowane malejąco prawdopodobieństwa),
   - wartość `p` (np. `0.9`).
2. Zaimplementuj funkcję:

```python
def nucleus_mask(probs_sorted, p):
    ...
```

która:

- zwraca wektor `mask` o tym samym kształcie, z wartościami 0 lub 1,
- `mask[i] = 1` jeśli indeks `i` należy do prefiksu `0..m`,
- `mask[i] = 0` w przeciwnym razie.
3. W `Answer`:

- policz `mask = nucleus_mask(probs_sorted, p)`,
- zapisz `final_answer` jako:

```text
m0,m1,m2,m3,m4
```

gdzie `mi` to `0` lub `1`, bez spacji.


In [34]:
#@title dane

import numpy as np

# Już posortowane malejąco
probs_sorted = np.array([0.5, 0.2, 0.15, 0.1, 0.05], dtype=float)
p = 0.8


In [35]:
#@title code

import numpy as np

def nucleus_mask(probs_sorted, p):
    """Zwraca maskę prefiksu dla nucleus (top-p) sampling."""
    cumsum = np.cumsum(probs_sorted)
    m = np.searchsorted(cumsum, p, side='left')
    
    mask = np.zeros_like(probs_sorted, dtype=int)
    mask[:m+1] = 1
    
    return mask


In [36]:
# @title Answer

mask = nucleus_mask(probs_sorted, p)
final_answer = ",".join(str(int(x)) for x in mask)
print(final_answer)

1,1,1,0,0


In [37]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD09"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


1,1,1,0,0
9 znaków
Wysłano!
Punkty za zadanie: 1


#Task D10

## 🧩 Log-likelihood sekwencji w modelu dekoderowym

W modelach typu GPT log-likelihood sekwencji \(x_1,\dots,x_T\) definiujemy jako:

$$
\log p(x_1,\dots,x_T) = \sum_{t=1}^T \log p(x_t \mid x_{<t}).
$$

W tym zadaniu policzysz log-likelihood krótkiej sekwencji
mając dane rozkłady `p(x_t | x_{<t})`.

---

## 🎯 Cel

- przećwiczyć pracę z logarytmami oraz sumowanie po krokach czasowych,
- zrozumieć, jak modele dekoderowe oceniają prawdopodobieństwo sekwencji.

---

## 📘 Kontekst

Mamy:

- sekwencję tokenów `tokens` długości `T`,
- macierz `probs_per_step` o kształcie `(T, vocab_size)`:

  - `probs_per_step[t, j] = p(token=j | kontekst do kroku t)`.

Wtedy:

- \(\log p(\text{sek}) = \sum_t \log p_t(\text{token}_t)\).

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `tokens` — wektor ID-ów długości `T`,
   - `probs_per_step` — macierz prawdopodobieństw.
2. Zaimplementuj funkcję:

```python
def sequence_log_likelihood(tokens, probs_per_step):
    ...
```

która:

- dla każdego kroku `t` wybiera odpowiednie `p_t = probs_per_step[t, tokens[t]]`,
- liczy sumę logarytmów naturalnych `np.log(p_t)`,
- zwraca wynik jako liczbę zmiennoprzecinkową (ujemną lub zero).
3. W `Answer`:

- policz `ll = sequence_log_likelihood(tokens, probs_per_step)`,
- zaokrąglij do **4 miejsc po przecinku**,
- zapisz `final_answer` jako string, np.:

```text
-3.2195
```


In [38]:
#@title dane

import numpy as np

# sekwencja długości 4, słownik rozmiaru 5
tokens = np.array([1, 2, 3, 1], dtype=int)

probs_per_step = np.array([
    [0.1, 0.6, 0.1, 0.1, 0.1],  # t=0 -> p(x0 | BOS)
    [0.2, 0.1, 0.5, 0.1, 0.1],  # t=1
    [0.3, 0.1, 0.1, 0.4, 0.1],  # t=2
    [0.25, 0.5, 0.05, 0.1, 0.1],# t=3
], dtype=float)


In [45]:
#@title code

import numpy as np

def sequence_log_likelihood(tokens, probs_per_step):
    """Zwraca log-likelihood sekwencji przy zadanych rozkładach krok po kroku."""
    log_likelihood = 0.0
    for t in range(len(tokens)):
        p_t = probs_per_step[t, tokens[t]]
        log_likelihood += np.log(p_t)
    return log_likelihood


In [46]:
# @title Answer

ll = sequence_log_likelihood(tokens, probs_per_step)
final_answer = f"{ll:.4f}"
print(final_answer)


-2.8134


In [47]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD10"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


-2.8134
7 znaków
Wysłano!
Punkty za zadanie: 1


#Task D11

## 🧩 Prosta analiza skalowania — regresja log-log

Modele dekoderowe (GPT-2, GPT-3) często analizuje się w układzie log-log:
log-strata w funkcji log-liczby parametrów.

W tym zadaniu dopasujesz **prostą linię** do kilku punktów w przestrzeni \((\log N, \log L)\).

---

## 🎯 Cel

- przećwiczyć logarytmowanie danych i regresję liniową,
- zrozumieć, skąd biorą się „prawa skalowania”.

---

## 📘 Kontekst

Mamy:

- `params` — wektor rozmiarów modeli \(N\),
- `losses` — odpowiadające im wartości straty \(L\).

Chcemy dopasować:

$$
\log L \approx a \cdot \log N + b.
$$

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `params` — np. `[1e6, 1e7, 1e8, 1e9]`,
   - `losses` — np. `[3.5, 3.0, 2.7, 2.5]`.
2. Zaimplementuj funkcję:

```python
def fit_scaling_law(params, losses):
    ...
```

która:

- liczy `x = np.log(params)`,
- liczy `y = np.log(losses)`,
- dopasowuje prostą metodą najmniejszych kwadratów (możesz użyć `np.polyfit(x, y, 1)`),
- zwraca współczynnik nachylenia `a`.
3. W `Answer`:

- policz `a = fit_scaling_law(params, losses)`,
- zaokrąglij do **4 miejsc po przecinku**,
- zapisz `final_answer` jako string, np.:

```text
-0.0800
```


In [50]:
#@title dane

import numpy as np

params = np.array([1e6, 1e7, 1e8, 1e9], dtype=float)
losses = np.array([3.5, 3.0, 2.7, 2.5], dtype=float)


In [51]:
#@title code

import numpy as np

def fit_scaling_law(params, losses):
    """Dopasowuje linię log-loss vs log-params i zwraca nachylenie a."""
    x = np.log(params)
    y = np.log(losses)
    a, b = np.polyfit(x, y, 1)
    return a


In [52]:
# @title Answer

a = fit_scaling_law(params, losses)
final_answer = f"{a:.4f}"
print(final_answer)

-0.0484


In [53]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD11"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


-0.0484
7 znaków
Wysłano!
Punkty za zadanie: 1


#Task D12

## 🧩 Ensemble dwóch modeli dekoderowych — mieszanie logitów

Czasem łączy się przewidywania dwóch modeli językowych,
np. GPT-2 i innego modelu, uśredniając ich logity.

W tym zadaniu zrobisz prosty ensemble dwóch modeli.

---

## 🎯 Cel

- poćwiczyć pracę na logitach i softmaxie,
- zrozumieć, jak łączyć rozkłady prawdopodobieństw z dwóch źródeł.

---

## 📘 Kontekst

Załóżmy, że mamy:

- `logits_A` — wektor logitów od modelu A,
- `logits_B` — wektor logitów od modelu B,
- skalary `alpha` i `beta`.

Tworzymy logity zespolone:

$$
\text{logits}_{\text{ens}} = \alpha \cdot \text{logits}_A + \beta \cdot \text{logits}_B,
$$

a następnie liczymy softmax, aby otrzymać rozkład prawdopodobieństwa.

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `logits_A`, `logits_B` (ten sam kształt),
   - `alpha`, `beta`.
2. Zaimplementuj funkcję:

```python
def ensemble_probs(logits_A, logits_B, alpha, beta):
    ...
```

która:

- liczy `logits_ens = alpha * logits_A + beta * logits_B`,
- liczy softmax po wektorze 1D,
- zwraca wektor prawdopodobieństw.
3. W `Answer`:

- policz `p = ensemble_probs(logits_A, logits_B, alpha, beta)`,
- zaokrąglij wartości do 3 miejsc po przecinku,
- zapisz w `final_answer` jako:

```text
p0,p1,p2,p3
```

bez spacji.


In [54]:
#@title dane

import numpy as np

logits_A = np.array([1.0, 0.0, -1.0, 0.5], dtype=float)
logits_B = np.array([0.5, 0.5, 0.0, -0.5], dtype=float)

alpha = 0.7
beta = 0.3


In [55]:
#@title code

import numpy as np

def ensemble_probs(logits_A, logits_B, alpha, beta):
    logits_ens = alpha * logits_A + beta * logits_B
    logits_ens = logits_ens - np.max(logits_ens)
    exp_logits = np.exp(logits_ens)
    return exp_logits / exp_logits.sum()


In [57]:
# @title Answer

p = ensemble_probs(logits_A, logits_B, alpha, beta)
rounded = [f"{v:.3f}" for v in p]
final_answer = ",".join(rounded)
print(final_answer)

0.448,0.223,0.095,0.234


In [58]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD12"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


0.448,0.223,0.095,0.234
23 znaków
Wysłano!
Punkty za zadanie: 1


#Task D13

## 🧩 T5 span corruption — wybór spanów do maskowania

W T5 pretraining polega na tzw. **span corruption**:
zamiast maskować pojedyncze tokeny jak w BERT, maskujemy **ciągłe fragmenty** (spany),
a na ich miejsce wstawiamy specjalne tokeny `<extra_id_0>`, `<extra_id_1>`, itd.

W tym zadaniu wybierzesz spany do maskowania w prostej, deterministycznej wersji.

---

## 🎯 Cel

- zrozumieć, że T5 maskuje całe fragmenty tekstu (spany), a nie pojedyncze tokeny,
- poćwiczyć operacje na indeksach i tworzenie list spanów.

---

## 📘 Kontekst

Załóżmy, że mamy sekwencję o długości `n` tokenów (indeksy `0..n-1`).
Chcemy wybrać **kolejne spany** o zadanych długościach, aż zabraknie miejsca.

Przykład:

- `n = 10`,
- `span_lengths = [2, 3, 2]`.

Algorytm:

1. Zaczynamy od `pos = 0` i pustej listy `spans`.
2. Dla każdej długości `L` w `span_lengths`:
   - jeśli `pos + L <= n`:
     - dodajemy span `(pos, pos+L)` (początek włącznie, koniec wyłącznie),
     - zwiększamy `pos = pos + L + gap`,
     - gdzie `gap` to przerwa między spanami (np. 1 token).
   - jeśli `pos + L > n`, przerywamy.
3. Otrzymujemy listę spanów np. `[(0,2), (3,6)]`.

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `n` — długość sekwencji,
   - `span_lengths` — listę długości spanów,
   - `gap` — odstęp (liczba tokenów) między kolejnymi spanami.
2. Zaimplementuj funkcję:

```python
def choose_spans(n, span_lengths, gap):
    ...
```

która zwraca **listę krotek** `(start, end)` zgodnie z algorytmem powyżej.
3. W `Answer`:

- wywołaj `spans = choose_spans(n, span_lengths, gap)`,
- zbuduj `final_answer` jako string:

```text
s0-e0;s1-e1;...
```

czyli każdy span zapisany jako `start-end`, a spany oddzielone średnikami.

Przykład formatu:

```text
0-2;3-6
```


In [59]:
#@title dane

n = 12
span_lengths = [2, 3, 2, 4]
gap = 1


In [60]:
#@title code

def choose_spans(n, span_lengths, gap):
    spans = []
    pos = 0
    for L in span_lengths:
        if pos + L <= n:
            spans.append((pos, pos + L))
            pos = pos + L + gap
        else:
            break
    return spans

In [61]:
# @title Answer

spans = choose_spans(n, span_lengths, gap)
parts = [f"{s}-{e}" for (s, e) in spans]
final_answer = ";".join(parts)
print(final_answer)

0-2;3-6;7-9


In [62]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD13"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


0-2;3-6;7-9
11 znaków
Wysłano!
Punkty za zadanie: 1


#Task D14

## 🧩 T5 span corruption — zbuduj sekwencje `input_ids` i `target_ids`

Kontynuujemy motyw T5: teraz na podstawie wybranych spanów masz zbudować:

- **wejście do enkodera** (z wstawionymi sentinelami),
- **docelową sekwencję dla dekodera** (ciąg fragmentów wyciętych, poprzedzanych sentinelami).

---

## 🎯 Cel

- zrozumieć dokładniej format pretrainingu T5,
- poćwiczyć manipulację tablicami i budowanie nowych sekwencji z fragmentów.

---

## 📘 Kontekst

Uproszczona wersja schematu T5:

- wejście (`encoder_input_ids`): oryginalna sekwencja, ale span `(s,e)` zastąpiony jednym tokenem `<extra_id_k>` (sentinel),
- cel (`decoder_target_ids`): sekwencja:

  `<extra_id_0>` + tokeny z pierwszego spanu + `<extra_id_1>` + tokeny z drugiego spanu + ...

W tym zadaniu:

- zakładamy, że tokeny to **liczby całkowite** (`np.array`),
- sentinel o indeksie `k` ma ID `SENTINEL_BASE - k` (jak w oryginalnym T5),
- spany są **niezachodzące i posortowane po `start`**.

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `input_ids` — oryginalną sekwencję tokenów,
   - `spans` — listę krotek `(start, end)`,
   - `SENTINEL_BASE` — np. `32000`.
2. Zaimplementuj funkcję:

```python
def t5_span_corruption(input_ids, spans, SENTINEL_BASE):
    ...
```

która zwraca krotkę `(encoder_input_ids, decoder_target_ids)` jako tablice `np.array`.
3. W `Answer`:

- policz `enc, dec = t5_span_corruption(input_ids, spans, SENTINEL_BASE)`,
- zbuduj `final_answer` w formacie:

```text
enc:ID0,ID1,...;dec:JD0,JD1,...
```

Przykład:

```text
enc:5,32000,9,10;dec:32000,7,8
```


In [63]:
#@title dane

import numpy as np

input_ids = np.array([5, 7, 8, 9, 10, 11], dtype=int)
# Span: wycinamy tokeny o indeksach [1,3) => 7,8
spans = [(1, 3)]
SENTINEL_BASE = 32000


In [64]:
#@title code

import numpy as np

def t5_span_corruption(input_ids, spans, SENTINEL_BASE):
    enc = []
    dec = []
    pos = 0
    for i, (s, e) in enumerate(spans):
        sentinel = SENTINEL_BASE - i
        enc.extend(input_ids[pos:s])
        enc.append(sentinel)
        dec.append(sentinel)
        dec.extend(input_ids[s:e])
        pos = e
    enc.extend(input_ids[pos:])
    return np.array(enc, dtype=int), np.array(dec, dtype=int)


In [65]:
# @title Answer

enc, dec = t5_span_corruption(input_ids, spans, SENTINEL_BASE)

enc_str = ",".join(str(int(x)) for x in enc)
dec_str = ",".join(str(int(x)) for x in dec)

final_answer = f"enc:{enc_str};dec:{dec_str}"
print(final_answer)

enc:5,32000,9,10,11;dec:32000,7,8


In [66]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD14"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


enc:5,32000,9,10,11;dec:32000,7,8
33 znaków
Wysłano!
Punkty za zadanie: 1


#Task D15

## 🧩 Cross-attention — macierz wag między enkoderem a dekoderem

W modelach encoder–decoder (np. T5) dekoder ma **cross-attention** do wyjść enkodera:
każda pozycja w dekoderze może zwracać uwagę na wszystkie pozycje enkodera.

W tym zadaniu policzysz **macierz wag cross-attention** dla pojedynczego headu.

---

## 🎯 Cel

- przećwiczyć liczenie `Q K^T` między dwoma różnymi sekwencjami,
- zrozumieć, że dekoder patrzy na reprezentacje z enkodera.

---

## 📘 Kontekst

Załóżmy:

- `Q_dec` — macierz zapytań z dekodera, kształt `(T_dec, d_k)`,
- `K_enc` — macierz kluczy z enkodera, kształt `(T_enc, d_k)`.

Wtedy macierz wag cross-attention:

$$
A = \text{softmax}\left(\frac{Q_{dec} K_{enc}^T}{\sqrt{d_k}}\right)
$$

gdzie softmax jest liczony **po kolumnach enkodera** dla każdej pozycji dekodera (czyli po osi odpowiadającej długości `T_enc`).

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `Q_dec` o kształcie `(2, 3)`,
   - `K_enc` o kształcie `(3, 3)`.
2. Zaimplementuj funkcję:

```python
def cross_attention_weights(Q_dec, K_enc):
    ...
```

która:

- liczy macierz `scores = Q_dec @ K_enc.T`,
- skaluje przez `sqrt(d_k)`,
- stosuje softmax po osi 1 (po kolumnach `K_enc`),
- zwraca macierz `A` o kształcie `(T_dec, T_enc)`.
3. W `Answer`:

- policz `A = cross_attention_weights(Q_dec, K_enc)`,
- spłaszcz do wektora row-major (najpierw cały wiersz 0, potem 1),
- zaokrąglij do 3 miejsc po przecinku,
- zapisz `final_answer` jako:

```text
a00,a01,a02,a10,a11,a12
```


In [67]:
#@title dane

import numpy as np

Q_dec = np.array([
    [0.5,  1.0,  0.0],
    [1.0, -0.5,  0.5],
], dtype=float)

K_enc = np.array([
    [0.5,  0.5,  0.0],
    [1.0,  0.0, -0.5],
    [0.0, -0.5,  1.0],
], dtype=float)


In [68]:
#@title code

import numpy as np

def cross_attention_weights(Q_dec, K_enc):
    d_k = Q_dec.shape[1]
    scores = Q_dec @ K_enc.T / np.sqrt(d_k)
    scores = scores - scores.max(axis=1, keepdims=True)
    exp = np.exp(scores)
    return exp / exp.sum(axis=1, keepdims=True)


In [70]:
# @title Answer

A = cross_attention_weights(Q_dec, K_enc)
flat = A.reshape(-1)
rounded = [f"{x:.3f}" for x in flat]
final_answer = ",".join(rounded)
print(final_answer)

0.425,0.368,0.207,0.273,0.364,0.364


In [71]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD15"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


0.425,0.368,0.207,0.273,0.364,0.364
35 znaków
Wysłano!
Punkty za zadanie: 1


#Task D16

## 🧩 Formatowanie zadań w stylu T5 — prompt text-to-text

T5 traktuje wszystkie zadania jako **tekst → tekst**.
Często stosuje się prosty format promptu, np.:

- `"sst2 sentence: <zdanie> sentiment: "`
- `"translate English to German: <zdanie>"`

W tym zadaniu zbudujesz prosty prompt do klasyfikacji i odpowiedź tekstową.

---

## 🎯 Cel

- uświadomić studentom, że wiele zadań da się zapisać jako mapping tekst→tekst,
- poćwiczyć prostą manipulację stringami.

---

## 📘 Kontekst

Załóżmy, że chcemy zadanie binary sentiment classification w stylu:

```text
sst2 sentence: <zdanie> sentiment:
```

Model ma odpowiedzieć:

- `"positive"` lub `"negative"`.

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `sentence` — zdanie (string),
   - `label` — etykietę numeryczną: `1` = pozytywne, `0` = negatywne.
2. Zaimplementuj funkcję:

```python
def build_t5_sentiment_io(sentence, label):
    ...
```

która zwraca krotkę `(inp, out)`:

- `inp` — prompt wejściowy, np. `"sst2 sentence: I love this movie sentiment:"`,
- `out` — oczekiwany tekst odpowiedzi: `"positive"` lub `"negative"`.
3. W `Answer`:

- wywołaj `inp, out = build_t5_sentiment_io(sentence, label)`,
- zbuduj `final_answer` jako:

```text
input:<TEKST_INP>;output:<TEKST_OUT>
```

Bez dodatkowych spacji na końcu.

Przykład:

```text
input:sst2 sentence: I love this movie sentiment:;output:positive
```


In [72]:
#@title dane

sentence = "The model works surprisingly well."
label = 1  # 1 = positive, 0 = negative


In [73]:
#@title code

def build_t5_sentiment_io(sentence, label):
    inp = f"sst2 sentence: {sentence} sentiment:"
    out = "positive" if label == 1 else "negative"
    return inp, out


In [74]:
# @title Answer

inp, out = build_t5_sentiment_io(sentence, label)
final_answer = f"input:{inp};output:{out}"
print(final_answer)

input:sst2 sentence: The model works surprisingly well. sentiment:;output:positive


In [75]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD16"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


input:sst2 sentence: The model works surprisingly well. sentiment:;output:positive
82 znaków
Wysłano!
Punkty za zadanie: 1


#Task D17

## 🧩 Prefiks zadania tłumaczeniowego (mT5) — wiele języków

W mT5 zadania tłumaczeniowe często zapisuje się jako tekst:

- `"translate <src_lang> to <tgt_lang>: <zdanie>"`

W tym zadaniu zbudujesz taki prefiks oraz oczekiwane wyjście.

---

## 🎯 Cel

- pokazać, jak wielojęzyczny model może dostawać zadania w jednym wspólnym formacie,
- poćwiczyć prostą logikę mapowania kodów językowych na nazwy.

---

## 📘 Kontekst

Załóżmy, że:

- `src_lang` i `tgt_lang` to kody `"en"`, `"de"`, `"pl"`, `"fr"`, itd.
- mamy słownik `lang2name`, np. `"en" -> "English"`, `"pl" -> "Polish"`.

Prompt ma mieć formę:

```text
translate <SrcName> to <TgtName>: <zdanie>
```

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `src_lang`, `tgt_lang`,
   - `sentence_src`,
   - słownik `lang2name`.
2. Zaimplementuj funkcję:

```python
def build_mt5_translate_io(src_lang, tgt_lang, sentence_src, sentence_tgt, lang2name):
    ...
```

która zwraca krotkę `(inp, out)`:

- `inp` — prompt jak wyżej,
- `out` — tekst docelowy `sentence_tgt`.
3. W `Answer`:

- wywołaj `inp, out = build_mt5_translate_io(...)`,
- zbuduj `final_answer`:

```text
input:<TEKST_INP>;output:<TEKST_OUT>
```


In [76]:
#@title dane

src_lang = "en"
tgt_lang = "pl"

sentence_src = "Transformers changed natural language processing."
sentence_tgt = "Transformatory zmieniły przetwarzanie języka naturalnego."

lang2name = {
    "en": "English",
    "pl": "Polish",
    "de": "German",
    "fr": "French",
}


In [77]:
#@title code

def build_mt5_translate_io(src_lang, tgt_lang, sentence_src, sentence_tgt, lang2name):
    inp = f"translate {lang2name[src_lang]} to {lang2name[tgt_lang]}: {sentence_src}"
    out = sentence_tgt
    return inp, out


In [78]:
# @title Answer

inp, out = build_mt5_translate_io(src_lang, tgt_lang, sentence_src, sentence_tgt, lang2name)
final_answer = f"input:{inp};output:{out}"
print(final_answer)

input:translate English to Polish: Transformers changed natural language processing.;output:Transformatory zmieniły przetwarzanie języka naturalnego.


In [79]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD17"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


input:translate English to Polish: Transformers changed natural language processing.;output:Transformatory zmieniły przetwarzanie języka naturalnego.
149 znaków
Wysłano!
Punkty za zadanie: 1


#Task D18

## 🧩 LongT5 — dzielenie dokumentu na segmenty z overlapem

Modele long-context, takie jak LongT5, często dzielą bardzo długie dokumenty
na **segmenty z nakładaniem się** (overlap), żeby zachować ciągłość kontekstu.

W tym zadaniu policzysz, jak wyznaczyć początki segmentów.

---

## 🎯 Cel

- przećwiczyć obliczanie indeksów segmentów dla long-context,
- zobaczyć, jak overlap zmniejsza „twarde cięcia” kontekstu.

---

## 📘 Kontekst

Załóżmy:

- dokument ma długość `doc_len` tokenów,
- model może przyjąć maksymalnie `max_len` tokenów na segment,
- kolejne segmenty przesuwamy o `stride` tokenów,
- zwykle `stride < max_len`, więc zachodzi overlap.

Segment `k` obejmuje tokeny o indeksach:

- od `start = k * stride`,
- do `end = start + max_len` (ale nie dalej niż `doc_len`).

Kończymy, gdy `start >= doc_len`.

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `doc_len`,
   - `max_len`,
   - `stride`.
2. Zaimplementuj funkcję:

```python
def make_long_segments(doc_len, max_len, stride):
    ...
```

która zwraca listę krotek `(start, end)` (end ekskluzywne).
3. W `Answer`:

- policz `segs = make_long_segments(doc_len, max_len, stride)`,
- zbuduj `final_answer` w formacie:

```text
s0-e0;s1-e1;...
```

np.:

```text
0-8;4-12;8-16
```


In [80]:
#@title dane

doc_len = 50
max_len = 16
stride = 8


In [81]:
#@title code

def make_long_segments(doc_len, max_len, stride):
    segments = []
    start = 0
    while start < doc_len:
        end = min(start + max_len, doc_len)
        segments.append((start, end))
        start += stride
    return segments


In [82]:
# @title Answer

segs = make_long_segments(doc_len, max_len, stride)
parts = [f"{s}-{e}" for (s, e) in segs]
final_answer = ";".join(parts)
print(final_answer)

0-16;8-24;16-32;24-40;32-48;40-50;48-50


In [83]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD18"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


0-16;8-24;16-32;24-40;32-48;40-50;48-50
39 znaków
Wysłano!
Punkty za zadanie: 1


#Task D19

## 🧩 Sliding-window attention — budowa maski

W wielu modelach long-context zamiast pełnej uwagi \(O(n^2)\) stosuje się **sliding-window attention**:
każdy token zwraca uwagę tylko na siebie oraz najbliższych sąsiadów.

---

## 🎯 Cel

- zbudować binarną maskę uwagi dla prostego schematu sliding-window,
- poćwiczyć indeksowanie i pętle.

---

## 📘 Kontekst

Załóżmy, że mamy sekwencję o długości `n`.
Definiujemy promień okna `w` (w tokenach).

Token na pozycji `i` może zwracać uwagę na token `j` wtedy i tylko wtedy, gdy:

- \(|i - j| \le w\).

Dodatkowo zawsze dopuszczamy uwagę na samego siebie (co i tak wynika z powyższego przy `w >= 0`).

Maskę zapiszemy jako macierz `M` o kształcie `(n, n)`:

- `M[i, j] = 1` jeśli **pozycja i może patrzeć na j**,
- `M[i, j] = 0` w przeciwnym razie.

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `n` — długość sekwencji,
   - `w` — promień okna.
2. Zaimplementuj funkcję:

```python
def sliding_window_mask(n, w):
    ...
```

która zwraca macierz `M` typu `np.int64` o kształcie `(n, n)` z wartościami 0/1.
3. W `Answer`:

- policz `M = sliding_window_mask(n, w)`,
- spłaszcz macierz do jednowymiarowego wektora w porządku row-major,
- zapisz `final_answer` jako ciąg 0/1 oddzielonych przecinkami, np.:

```text
1,1,0,0,0,1,1,1,0,0,0,1,1,0,0,0,1,1,0,0,0,1,1,1,0
```

bez spacji.


In [84]:
#@title dane

import numpy as np

n = 5
w = 1


In [85]:
#@title code

import numpy as np

def sliding_window_mask(n, w):
    M = np.zeros((n, n), dtype=np.int64)
    for i in range(n):
        start = max(0, i - w)
        end = min(n, i + w + 1)
        M[i, start:end] = 1
    return M


In [86]:
# @title Answer

M = sliding_window_mask(n, w)
flat = M.reshape(-1)
final_answer = ",".join(str(int(x)) for x in flat)
print(final_answer)

1,1,0,0,0,1,1,1,0,0,0,1,1,1,0,0,0,1,1,1,0,0,0,1,1


In [87]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD19"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


1,1,0,0,0,1,1,1,0,0,0,1,1,1,0,0,0,1,1,1,0,0,0,1,1
49 znaków
Wysłano!
Punkty za zadanie: 1


#Task D20

## 🧩 Longformer-style attention — lokalna + globalna uwaga dla jednego tokenu

Longformer łączy **lokalną uwagę okienkową** z **globalnymi tokenami**,
które mogą patrzeć na wszystkich i są widziane przez wszystkich.

W tym zadaniu zbudujesz maskę uwagi dla **jednego tokenu** w takim schemacie.

---

## 🎯 Cel

- zrozumieć, jak łączą się lokalne okna i globalne pozycje,
- poćwiczyć operacje na indexach.

---

## 📘 Kontekst

Załóżmy:

- sekwencja długości `n`,
- promień lokalnego okna `w`,
- zbiór indeksów globalnych `global_indices` (np. `[0, 3]`),
- rozważamy **konkretny token** o indeksie `i`.

Token `i` może patrzeć na pozycję `j` jeśli:

1. pozycja `j` jest w lokalnym oknie: \(|i-j| \le w\),
   **lub**
2. `j` jest globalna (`j` w `global_indices`),
   **lub**
3. `i` sam jest globalny — wtedy może patrzeć na wszystkie pozycje.

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `n`, `w`,
   - `global_indices` (lista int),
   - `i` — indeks tokenu, dla którego liczymy maskę.
2. Zaimplementuj funkcję:

```python
def longformer_row_mask(n, w, global_indices, i):
    ...
```

która zwraca wektor długości `n` z 0/1.
3. W `Answer`:

- policz `row = longformer_row_mask(n, w, global_indices, i)`,
- zapisz `final_answer` jako `r0,r1,...,r_{n-1}` (0/1, bez spacji).


In [88]:
#@title dane

import numpy as np

n = 8
w = 1
global_indices = [0, 4]
i = 4  # token globalny


In [89]:
#@title code

import numpy as np

def longformer_row_mask(n, w, global_indices, i):
    row = np.zeros(n, dtype=np.int64)
    if i in global_indices:
        row[:] = 1
    else:
        start = max(0, i - w)
        end = min(n, i + w + 1)
        row[start:end] = 1
        row[global_indices] = 1
    return row



In [90]:
# @title Answer

row = longformer_row_mask(n, w, global_indices, i)
final_answer = ",".join(str(int(x)) for x in row)
print(final_answer)

1,1,1,1,1,1,1,1


In [91]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD20"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


1,1,1,1,1,1,1,1
15 znaków
Wysłano!
Punkty za zadanie: 1


#Task D21

## 🧩 Ile jest połączeń uwagi? Full vs sliding-window vs Longformer

W tym zadaniu porównasz **liczbę połączeń attention** (niezerowych wpisów w macierzy)
w trzech wariantach:

- pełna uwaga,
- sliding-window,
- Longformer (okno + kilka globalnych pozycji).

---

## 🎯 Cel

- zrozumieć różnice złożoności między różnymi schematami uwagi,
- policzyć dokładne liczby połączeń dla małego przykładu.

---

## 📘 Kontekst

Dla sekwencji długości `n`:

- pełna uwaga: \(n^2\) połączeń (każdy token może patrzeć na każdy),
- sliding-window o promieniu `w`:
  - przybliżenie: każdy token ma \(1 + 2w\) połączeń (z uwzględnieniem brzegów),
- Longformer z:
  - promieniem `w`,
  - `g` globalnymi tokenami,
  - zakładamy, że:
    - każdy globalny token może patrzeć na wszystkie `n` pozycji,
    - każdy token może patrzeć na wszystkie `g` globalne,
    - oraz ma lokalne połączenia sliding-window.

Dla Longformera policzymy:

- `local_edges` — liczba połączeń z samego sliding-window,
- `global_to_all` — `g * n`,
- `all_to_global` — `n * g`,
- suma to przybliżona liczba połączeń.

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `n`, `w`, `g`.
2. Zaimplementuj funkcję:

```python
def count_full_edges(n):
    ...
def count_sliding_edges(n, w):
    ...
def count_longformer_edges(n, w, g):
    ...
```

zgodnie z opisem powyżej (w sliding-window możesz użyć przybliżenia `n * (1 + 2*w)`).
3. W `Answer`:

- policz trzy liczby: `full_edges`, `sliding_edges`, `longformer_edges`,
- zapisz `final_answer` jako:

```text
full=LICZBA1;sliding=LICZBA2;longformer=LICZBA3
```


In [92]:
#@title dane

n = 512
w = 4
g = 8


In [93]:
#@title code

def count_full_edges(n):
    return n * n

def count_sliding_edges(n, w):
    return n * (1 + 2 * w)

def count_longformer_edges(n, w, g):
    local_edges = n * (1 + 2 * w)
    global_to_all = g * n
    all_to_global = n * g
    return local_edges + global_to_all + all_to_global


In [94]:
# @title Answer

full_edges = count_full_edges(n)
sliding_edges = count_sliding_edges(n, w)
longformer_edges = count_longformer_edges(n, w, g)

final_answer = f"full={full_edges};sliding={sliding_edges};longformer={longformer_edges}"
print(final_answer)

full=262144;sliding=4608;longformer=12800


In [95]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD21"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


full=262144;sliding=4608;longformer=12800
41 znaków
Wysłano!
Punkty za zadanie: 1


#Task D22

## 🧩 Efektywny zasięg kontekstu po wielu warstwach sliding-window

Nawet jeśli pojedyncza warstwa ma tylko lokalne okno uwagi,
to po przejściu przez **wiele warstw** efektywny zasięg informacji rośnie.

W przybliżeniu po `L` warstwach sliding-window o promieniu `w`
token może „pośrednio” zobaczyć tokeny oddalone o \(L \cdot w\).

---

## 🎯 Cel

- wyprowadzić prosty wzór na efektywny promień kontekstu,
- poćwiczyć elementarną arytmetykę w funkcji parametrów.

---

## 📘 Kontekst

Przyjmujemy uproszczony model:

- każda warstwa ma sliding-window o promieniu `w`,
- informacja może przepływać co warstwę o `w` tokenów w każdą stronę,
- po `L` warstwach efektywny promień to:

$$
r_{\text{eff}} = L \cdot w.
$$

Jeśli chcemy pokryć całą sekwencję długości `n`,
wystarczy, by \(r_{\text{eff}} \ge \frac{n-1}{2}\) (dla środkowego tokenu).

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `n`, `w`, `L`.
2. Zaimplementuj funkcje:

```python
def effective_radius(L, w):
    ...
def covers_full_sequence(n, L, w):
    ...
```

gdzie:

- `effective_radius` zwraca \(L \cdot w\),
- `covers_full_sequence` zwraca `True`/`False`, czy ten promień wystarcza, by dla **środkowego tokenu**
  sięgnąć do obu końców sekwencji (skorzystaj z warunku \(r_{eff} \ge (n-1)/2\)).
3. W `Answer`:

- policz `r_eff = effective_radius(L, w)`,
- policz `covers = covers_full_sequence(n, L, w)`,
- zapisz `final_answer` jako:

```text
radius=R;covers=BOOL
```

gdzie `BOOL` to `True` lub `False`.


In [96]:
#@title dane

n = 129
w = 4
L = 10


In [97]:
#@title code

def effective_radius(L, w):
    return L * w

def covers_full_sequence(n, L, w):
    r_eff = effective_radius(L, w)
    return r_eff >= (n - 1) / 2



In [98]:
# @title Answer

r_eff = effective_radius(L, w)
covers = covers_full_sequence(n, L, w)

final_answer = f"radius={r_eff};covers={covers}"
print(final_answer)

radius=40;covers=False


In [99]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD22"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


radius=40;covers=False
22 znaków
Wysłano!
Punkty za zadanie: 1


#Task D23

## 🧩 Blokowy podział sekwencji — przygotowanie do FlashAttention

Implementacje w stylu FlashAttention dzielą sekwencję na **bloki** (tiles),
które można przetwarzać w kawałkach na GPU.

W tym zadaniu policzysz, jak wygląda taki podział dla prostej sekwencji.

---

## 🎯 Cel

- zrozumieć ideę dzielenia sekwencji na bloki,
- poćwiczyć arytmetykę dzielenia z resztą.

---

## 📘 Kontekst

Załóż, że:

- sekwencja ma długość `n`,
- dzielimy ją na bloki długości `block_size`,
- ostatni blok może być krótszy, jeśli `n` nie jest wielokrotnością `block_size`.

Chcemy znać:

- `num_blocks` — ile bloków powstanie,
- `last_block_size` — długość ostatniego bloku.

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `n`,
   - `block_size`.
2. Zaimplementuj funkcję:

```python
def block_stats(n, block_size):
    ...
```

która zwraca `(num_blocks, last_block_size)`.
3. W `Answer`:

- policz `num_blocks, last_block_size = block_stats(n, block_size)`,
- zapisz `final_answer` jako:

```text
blocks=B;last=L
```


In [100]:
#@title dane

n = 102
block_size = 16


In [101]:
#@title code

def block_stats(n, block_size):
    num_blocks = (n + block_size - 1) // block_size
    last_block_size = n % block_size
    if last_block_size == 0:
        last_block_size = block_size
    return num_blocks, last_block_size

In [102]:
# @title Answer

num_blocks, last_block_size = block_stats(n, block_size)
final_answer = f"blocks={num_blocks};last={last_block_size}"
print(final_answer)

blocks=7;last=6


In [103]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD23"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


blocks=7;last=6
15 znaków
Wysłano!
Punkty za zadanie: 1


#Task D24

## 🧩 FLOPs na warstwę — full attention vs „szybsza” implementacja

Załóżmy bardzo uproszczony model kosztu:

- klasyczna implementacja attention: \(C_{full} \cdot n^2\),
- zoptymalizowana implementacja (np. FlashAttention): \(C_{flash} \cdot n^2\),

gdzie:

- \(n\) — długość sekwencji,
- \(C_{full}\), \(C_{flash}\) — stałe współczynniki (np. związane z ruchem pamięci).

W tym zadaniu policzysz, **ile razy** szybsza jest wersja zoptymalizowana dla zadanych parametrów.

---

## 🎯 Cel

- uświadomić, że nawet przy tej samej złożoności \(O(n^2)\) można sporo zyskać na stałych,
- poćwiczyć proste obliczenia na liczbach zmiennoprzecinkowych.

---

## 📘 Kontekst

Przyjmujemy:

- `flops_full = C_full * n**2`,
- `flops_flash = C_flash * n**2`,
- interesuje nas stosunek:

$$
\text{speedup} = \frac{\text{flops}_{full}}{\text{flops}_{flash}}.
$$

---

## ✅ Twoje zadanie

1. W `dane` masz:
   - `n`,
   - `C_full`,
   - `C_flash`.
2. Zaimplementuj funkcję:

```python
def attention_speedup(n, C_full, C_flash):
    ...
```

która zwraca wartość `speedup` jako liczbę zmiennoprzecinkową.
3. W `Answer`:

- policz `s = attention_speedup(n, C_full, C_flash)`,
- zaokrąglij do 3 miejsc po przecinku,
- zapisz `final_answer` jako string, np.:

```text
1.875
```


In [104]:
#@title dane

n = 2048
C_full = 2.0
C_flash = 0.8


In [105]:
#@title code

def attention_speedup(n, C_full, C_flash):
    return C_full / C_flash

In [106]:
# @title Answer

s = attention_speedup(n, C_full, C_flash)
final_answer = f"{s:.3f}"
print(final_answer)

2.500


In [107]:
#@title Submit answer (final_answer)
import requests
taskID = "TaskD24"
url = 'https://www.duszekjk.com/ugpt/api/submit_answer/'
final_answer_size = len(final_answer)

final_answer_send = final_answer[:300] + "\n" + str(final_answer_size) + " znaków"
print(final_answer_send)
data = {
    'student_id': Student_ID,
    'student_mail': Mail,
    'task': taskID,
    'grupa': Grupa,
    'answer': final_answer_send,
    'share_link': Link_to_this_colab
}

if Link_to_this_colab == colab_base or Link_to_this_colab == "":
    print("Podaj link do swojego Colab")
    display(Javascript('alert("Podaj link do swojego Colab");'))
else:
    if final_answer == "":
        print("Podaj rozwiązanie")
        display(Javascript('alert("Podaj rozwiązanie");'))
    else:
        response = requests.post(url, json=data)
        if response.status_code == 200:
            response_data = response.json()
            print("Wysłano!")

            points = response_data.get('points', 0)
            if points > 0:
                print(f"Punkty za zadanie: {points}")
            else:
                print("Zadanie wymaga ręcznego zatwierdzenia lub wynik jest niepoprawny")
        else:
            print("błąd wysyłki:", response.status_code)
            print("Response Text:", response.text)


2.500
5 znaków
Wysłano!
Punkty za zadanie: 1
